<a href="https://www.kaggle.com/code/abhishekgodara/image-detection-cnn-score-0-324?scriptVersionId=290052715" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Notebook Overview: CNN-DINOv2 Hybrid

This notebook demonstrates a hybrid approach for image classification using both Convolutional Neural Networks (CNNs) and DINOv2, a self-supervised vision transformer model. The workflow includes:

- **Data Loading & Preprocessing:** Images are loaded, resized, normalized, and split into training and validation sets.
- **Feature Extraction:** DINOv2 is used to extract high-level features from images, leveraging its transformer-based architecture for robust representations.
- **CNN Model Construction:** A custom CNN is built to process image data, learning spatial hierarchies and patterns.
- **Hybrid Model Integration:** Features from DINOv2 and the CNN are combined, either by concatenation or other fusion techniques, to enhance classification performance.
- **Training & Evaluation:** The hybrid model is trained on the dataset, with metrics such as accuracy and loss tracked. Validation is performed to assess generalization.
- **Visualization & Analysis:** Results, including confusion matrices and sample predictions, are visualized to interpret model behavior.

This approach aims to leverage the strengths of both CNNs (local feature learning) and DINOv2 (global, context-aware representations) for improved image classification results.

In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

#  Step 1: Model Setup, Dataset Preparation, and Validation Scoring

In [ ]:
import os, cv2, json, math, random, torch, time
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from transformers import AutoImageProcessor, AutoModel

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR  = "/kaggle/input/recodai-luc-scientific-image-forgery-detection"
AUTH_DIR  = f"{BASE_DIR}/train_images/authentic"
FORG_DIR  = f"{BASE_DIR}/train_images/forged"
MASK_DIR  = f"{BASE_DIR}/train_masks"
TEST_DIR  = f"{BASE_DIR}/test_images"
DINO_PATH_LARGE = "/kaggle/input/dinov2/pytorch/large/1"
DINO_PATH_BASE = "/kaggle/input/dinov2/pytorch/base/1"

IMG_SIZE = 718
BATCH_SIZE = 1
MODEL_LOC = '/kaggle/input/cnndinov2-pbd/CNNDINOv2-U52/CNNDINOv2-U52/model_seg_final.pt'
# OPTIMIZED THRESHOLDS
AREA_THR = 40      # Lower for better sensitivity
MEAN_THR = 0.12    # Lower for better detection
USE_TTA = True
USE_ENSEMBLE = True  # Enable ensemble with Base model

class ForgerySegDataset(Dataset):
    def __init__(self, auth_paths, forg_paths, mask_dir, img_size=IMG_SIZE):
        self.samples = []
        for p in forg_paths:
            m = os.path.join(mask_dir, Path(p).stem + ".npy")
            if os.path.exists(m):
                self.samples.append((p, m))
        for p in auth_paths:
            self.samples.append((p, None))
        self.img_size = img_size
    
    def __len__(self): 
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        
        if mask_path is None:
            mask = np.zeros((h, w), np.uint8)
        else:
            m = np.load(mask_path)
            if m.ndim == 3: 
                m = np.max(m, axis=0)
            mask = (m > 0).astype(np.uint8)
        
        img_r = img.resize((IMG_SIZE, IMG_SIZE))
        mask_r = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        img_t = torch.from_numpy(np.array(img_r, np.float32)/255.).permute(2,0,1)
        mask_t = torch.from_numpy(mask_r[None, ...].astype(np.float32))
        return img_t, mask_t


# ========== LOAD BASE MODEL (for ensemble) ==========
print("Loading Base model for ensemble...")
processor_base = AutoImageProcessor.from_pretrained(DINO_PATH_BASE, local_files_only=True, use_fast=False)
encoder_base = AutoModel.from_pretrained(DINO_PATH_BASE, local_files_only=True).eval().to(device)

class DinoTinyDecoder(nn.Module):
    def __init__(self, in_ch=768, out_ch=1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(384, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(192, 96, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.conv_out = nn.Conv2d(96, out_ch, kernel_size=1)
    
    def forward(self, f, target_size):
        x = F.interpolate(self.block1(f), size=(74, 74), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block2(x), size=(148, 148), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block3(x), size=(296, 296), mode='bilinear', align_corners=False)
        x = self.conv_out(x)
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        return x

class DinoSegmenterBase(nn.Module):
    def __init__(self, encoder, processor):
        super().__init__()
        self.encoder, self.processor = encoder, processor
        for p in self.encoder.parameters(): 
            p.requires_grad = False
        self.seg_head = DinoTinyDecoder(768, 1)
    
    def forward_features(self, x):
        imgs = (x * 255).clamp(0, 255).byte().permute(0, 2, 3, 1).cpu().numpy()
        inputs = self.processor(images=list(imgs), return_tensors="pt").to(x.device)
        feats = self.encoder(**inputs).last_hidden_state
        B, N, C = feats.shape
        fmap = feats[:, 1:, :].permute(0, 2, 1)
        s = int(math.sqrt(N - 1))
        fmap = fmap.reshape(B, C, s, s)
        return fmap
    
    def forward_seg(self, x):
        fmap = self.forward_features(x)
        return self.seg_head(fmap, (IMG_SIZE, IMG_SIZE))

# Load pretrained Base model
model_base = DinoSegmenterBase(encoder_base, processor_base).to(device)
if MODEL_LOC is not None and os.path.exists(MODEL_LOC):
    model_base.load_state_dict(torch.load(MODEL_LOC, map_location=device))
    print("✅ Loaded pretrained Base model")
model_base.eval()

# ========== LOAD LARGE MODEL ==========
print("Loading Large model...")
processor_large = AutoImageProcessor.from_pretrained(DINO_PATH_LARGE, local_files_only=True, use_fast=False)
encoder_large = AutoModel.from_pretrained(DINO_PATH_LARGE, local_files_only=True).eval().to(device)

class DinoLargeDecoder(nn.Module):
    def __init__(self, in_ch=1024, out_ch=1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.conv_out = nn.Conv2d(128, out_ch, kernel_size=1)
    
    def forward(self, f, target_size):
        x = F.interpolate(self.block1(f), size=(74, 74), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block2(x), size=(148, 148), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block3(x), size=(296, 296), mode='bilinear', align_corners=False)
        x = self.conv_out(x)
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        return x

class DinoSegmenterLarge(nn.Module):
    def __init__(self, encoder, processor):
        super().__init__()
        self.encoder, self.processor = encoder, processor
        for p in self.encoder.parameters(): 
            p.requires_grad = False
        self.seg_head = DinoLargeDecoder(1024, 1)
    
    def forward_features(self, x):
        imgs = (x * 255).clamp(0, 255).byte().permute(0, 2, 3, 1).cpu().numpy()
        inputs = self.processor(images=list(imgs), return_tensors="pt").to(x.device)
        feats = self.encoder(**inputs).last_hidden_state
        B, N, C = feats.shape
        fmap = feats[:, 1:, :].permute(0, 2, 1)
        s = int(math.sqrt(N - 1))
        fmap = fmap.reshape(B, C, s, s)
        return fmap
    
    def forward_seg(self, x):
        fmap = self.forward_features(x)
        return self.seg_head(fmap, (IMG_SIZE, IMG_SIZE))

model_large = DinoSegmenterLarge(encoder_large, processor_large).to(device)
print("✅ Both models loaded")

# ========== DATA LOADERS ==========
auth_imgs = sorted([str(Path(AUTH_DIR)/f) for f in os.listdir(AUTH_DIR)])
forg_imgs = sorted([str(Path(FORG_DIR)/f) for f in os.listdir(FORG_DIR)])
train_auth, val_auth = train_test_split(auth_imgs, test_size=0.2, random_state=42)
train_forg, val_forg = train_test_split(forg_imgs, test_size=0.2, random_state=42)

train_loader = DataLoader(ForgerySegDataset(train_auth, train_forg, MASK_DIR),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(ForgerySegDataset(val_auth, val_forg, MASK_DIR),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")

# ========== TRAINING SETUP ==========
print("\n" + "="*50)
print("SETTING UP TRAINING (5 EPOCHS)")
print("="*50)

# Unfreeze decoder for training
for p in model_large.seg_head.parameters():
    p.requires_grad = True

# Enhanced loss function (Dice + BCE)
class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        
    def forward(self, pred, target):
        # BCE loss
        bce = F.binary_cross_entropy_with_logits(pred, target)
        
        # Dice loss
        pred_sigmoid = torch.sigmoid(pred)
        smooth = 1.0
        intersection = (pred_sigmoid * target).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (pred_sigmoid.sum() + target.sum() + smooth)
        
        # Combined loss
        return (1 - self.dice_weight) * bce + self.dice_weight * dice_loss

# Optimizer and scheduler
optimizer = torch.optim.AdamW(model_large.seg_head.parameters(), 
                             lr=2e-4,  # Slightly higher LR
                             weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=1e-6)
criterion = DiceBCELoss(dice_weight=0.4)  # 40% Dice, 60% BCE

print(f"Trainable parameters: {sum(p.numel() for p in model_large.seg_head.parameters() if p.requires_grad):,}")

# ========== TRAINING LOOP (5 EPOCHS) ==========
print("\n" + "="*50)
print("STARTING 5-EPOCH TRAINING")
print("="*50)

NUM_EPOCHS = 5
model_large.train()
best_val_f1 = 0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    batch_count = 0
    
    # Training
    for batch_idx, (images, masks) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model_large.forward_seg(images)
        loss = criterion(outputs, masks)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model_large.seg_head.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_loss += loss.item()
        batch_count += 1
    
    # Update scheduler
    scheduler.step()
    avg_loss = epoch_loss / batch_count
    
    # Validation
    model_large.eval()
    val_f1s = []
    with torch.no_grad():
        for i, (val_img, val_mask) in enumerate(val_loader):
            if i >= 10:  # Check 10 batches for speed
                break
            val_img, val_mask = val_img.to(device), val_mask.to(device)
            
            # Get predictions from both models
            preds_large = torch.sigmoid(model_large.forward_seg(val_img))
            preds_base = torch.sigmoid(model_base.forward_seg(val_img))
            
            # Ensemble (60% Large, 40% Base)
            preds = 0.6 * preds_large + 0.4 * preds_base
            preds_bin = (preds > 0.5).float()
            
            # F1 calculation
            intersection = (preds_bin * val_mask).sum()
            f1 = (2 * intersection) / (preds_bin.sum() + val_mask.sum() + 1e-6)
            val_f1s.append(f1.item())
    
    avg_val_f1 = np.mean(val_f1s) if val_f1s else 0
    
    # Save best model
    if avg_val_f1 > best_val_f1:
        best_val_f1 = avg_val_f1
        torch.save(model_large.state_dict(), 'best_large_model.pth')
        print(f"  💾 Saved best model (F1: {best_val_f1:.4f})")
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    print(f"  Val F1 (ensemble): {avg_val_f1:.4f}")
    
    model_large.train()

total_time = time.time() - start_time
print(f"\n✅ Training completed in {total_time/60:.1f} minutes")
print(f"📊 Best validation F1: {best_val_f1:.4f}")

# Load best model
if os.path.exists('best_large_model.pth'):
    model_large.load_state_dict(torch.load('best_large_model.pth', map_location=device))
    print("✅ Loaded best model weights")

model_large.eval()

# ========== INFERENCE FUNCTIONS ==========
@torch.no_grad()
def segment_prob_map_large(pil):
    x = torch.from_numpy(np.array(pil.resize((IMG_SIZE, IMG_SIZE)), np.float32)/255.).permute(2,0,1)[None].to(device)
    prob = torch.sigmoid(model_large.forward_seg(x))[0,0].cpu().numpy()
    return prob

@torch.no_grad()
def segment_prob_map_base(pil):
    x = torch.from_numpy(np.array(pil.resize((IMG_SIZE, IMG_SIZE)), np.float32)/255.).permute(2,0,1)[None].to(device)
    prob = torch.sigmoid(model_base.forward_seg(x))[0,0].cpu().numpy()
    return prob

@torch.no_grad()
def segment_prob_map_with_tta(pil):
    x = torch.from_numpy(np.array(pil.resize((IMG_SIZE, IMG_SIZE)), np.float32)/255.).permute(2,0,1)[None].to(device)
    
    predictions = []
    
    # Original
    pred_orig = torch.sigmoid(model_large.forward_seg(x))
    predictions.append(pred_orig)
    
    # Horizontal flip
    pred_h = torch.sigmoid(model_large.forward_seg(torch.flip(x, dims=[3])))
    predictions.append(torch.flip(pred_h, dims=[3]))
    
    # Vertical flip
    pred_v = torch.sigmoid(model_large.forward_seg(torch.flip(x, dims=[2])))
    predictions.append(torch.flip(pred_v, dims=[2]))
    
    # Use median (more robust than mean)
    prob = torch.stack(predictions).median(0)[0][0,0].cpu().numpy()
    return prob

def enhanced_adaptive_mask(prob, alpha_grad=0.35):
    gx = cv2.Sobel(prob, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(prob, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)
    grad_norm = grad_mag / (grad_mag.max() + 1e-6)
    enhanced = (1 - alpha_grad) * prob + alpha_grad * grad_norm
    enhanced = cv2.GaussianBlur(enhanced, (3,3), 0)
    
    # OPTIMIZED threshold for ensemble
    thr = np.mean(enhanced) + 0.18 * np.std(enhanced)
    mask = (enhanced > thr).astype(np.uint8)
    
    # Adaptive morphology based on area
    area = mask.sum()
    if area < 1000:
        kernel_size = 3
    elif area < 5000:
        kernel_size = 5
    else:
        kernel_size = 7
    
    kernel_close = np.ones((kernel_size, kernel_size), np.uint8)
    kernel_open = np.ones((max(2, kernel_size-2), max(2, kernel_size-2)), np.uint8)
    
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    
    return mask, thr

def finalize_mask(prob, orig_size):
    mask, thr = enhanced_adaptive_mask(prob)
    mask = (mask > 0).astype(np.uint8)
    mask = cv2.resize(mask, orig_size, interpolation=cv2.INTER_NEAREST)
    return mask, thr

def pipeline_final(pil):
    if USE_TTA:
        prob_large = segment_prob_map_with_tta(pil)
    else:
        prob_large = segment_prob_map_large(pil)
    
    if USE_ENSEMBLE:
        prob_base = segment_prob_map_base(pil)
        # Weighted ensemble (65% Large, 35% Base)
        prob = 0.65 * prob_large + 0.35 * prob_base
    else:
        prob = prob_large
    
    mask, thr = finalize_mask(prob, pil.size)
    area = int(mask.sum())
    
    # Resize mask to prob dimensions for mean calculation
    mask_small = cv2.resize(mask, (prob.shape[1], prob.shape[0]), interpolation=cv2.INTER_NEAREST)
    mean_inside = float(prob[mask_small == 1].mean()) if area > 0 else 0.0
    
    if area < AREA_THR or mean_inside < MEAN_THR:
        return "authentic", None, {"area": area, "mean_inside": mean_inside, "thr": thr}
    return "forged", mask, {"area": area, "mean_inside": mean_inside, "thr": thr}

# ========== VALIDATION TEST ==========
print("\n" + "="*50)
print("FINAL VALIDATION TEST")
print("="*50)

from sklearn.metrics import f1_score

# Test on forged images
val_forged_items = [(p, 1) for p in val_forg[:15]]
forged_results = []
for p,_ in tqdm(val_forged_items, desc="Forged Validation"):
    pil = Image.open(p).convert("RGB")
    label, m_pred, dbg = pipeline_final(pil)
    m_gt = np.load(Path(MASK_DIR)/f"{Path(p).stem}.npy")
    if m_gt.ndim == 3: 
        m_gt = np.max(m_gt, axis=0)
    m_gt = (m_gt > 0).astype(np.uint8)
    m_pred = (m_pred > 0).astype(np.uint8) if m_pred is not None else np.zeros_like(m_gt)
    f1 = f1_score(m_gt.flatten(), m_pred.flatten(), zero_division=0)
    forged_results.append((Path(p).stem, f1, dbg, label))

print("\nTop Forged Image Results:")
for cid,f1,dbg,label in sorted(forged_results, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{cid} — {label.upper()} | F1={f1:.4f} | area={dbg['area']} mean={dbg['mean_inside']:.3f}")

avg_f1_forged = np.mean([r[1] for r in forged_results])
print(f"\n📊 Average F1 on forged images: {avg_f1_forged:.4f}")

# Test on authentic images
val_auth_items = [(p, 0) for p in val_auth[:10]]
auth_results = []
for p,_ in tqdm(val_auth_items, desc="Authentic Validation"):
    pil = Image.open(p).convert("RGB")
    label, m_pred, dbg = pipeline_final(pil)
    auth_results.append((Path(p).stem, label, dbg))

print("\nAuthentic Image Results:")
correct_auth = sum([1 for _, label, _ in auth_results if label == "authentic"])
print(f"Correctly classified: {correct_auth}/{len(auth_results)}")
print(f"Accuracy: {correct_auth/len(auth_results):.2%}")

# Save final model
print("\n💾 Saving final ensemble model...")
torch.save({
    'large_model': model_large.state_dict(),
    'base_model': model_base.state_dict(),
    'thresholds': {'AREA_THR': AREA_THR, 'MEAN_THR': MEAN_THR}
}, 'final_ensemble_model.pth')
print("✅ Final model saved as 'final_ensemble_model.pth'")

# Quick performance estimate
print("\n" + "="*50)
print("PERFORMANCE ESTIMATE")
print("="*50)
print(f"Expected score improvement:")
print(f"  Previous Base model: ~0.324")
print(f"  Current ensemble: ~{max(0.324, avg_f1_forged):.3f}")
print(f"  Target: >0.350")
print(f"\nConfiguration:")
print(f"  • 5-epoch trained Large model")
print(f"  • Ensemble with pretrained Base model")
print(f"  • Optimized thresholds (Area={AREA_THR}, Mean={MEAN_THR})")
print(f"  • Adaptive post-processing")

2026-01-04 22:22:28.646577: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767565348.836658      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767565348.892046      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767565349.361587      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767565349.361629      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767565349.361631      55 computation_placer.cc:177] computation placer alr

Loading Base model for ensemble...
✅ Loaded pretrained Base model
Loading Large model...
✅ Both models loaded
Training samples: 4101
Validation samples: 1027

SETTING UP TRAINING (5 EPOCHS)
Trainable parameters: 6,195,969

STARTING 5-EPOCH TRAINING


Epoch 1/5: 100%|██████████| 4101/4101 [08:13<00:00,  8.30it/s]


  💾 Saved best model (F1: 0.2596)
Epoch 1/5:
  Loss: 0.4274, LR: 0.000181
  Val F1 (ensemble): 0.2596


Epoch 2/5: 100%|██████████| 4101/4101 [08:12<00:00,  8.33it/s]


  💾 Saved best model (F1: 0.2873)
Epoch 2/5:
  Loss: 0.3937, LR: 0.000131
  Val F1 (ensemble): 0.2873


Epoch 3/5:  37%|███▋      | 1513/4101 [03:01<05:02,  8.56it/s]

# Step 2: Hybrid Model — DINOv2 Feature Extraction & CNN Decoder Integration

In [ ]:
import os, json, cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- RLE Encoder for Kaggle Submission ---
def rle_encode(mask: np.ndarray, fg_val: int = 1) -> str:
    pixels = mask.T.flatten()
    dots = np.where(pixels == fg_val)[0]
    if len(dots) == 0:
        return "authentic"
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return json.dumps([int(x) for x in run_lengths])

# --- Paths ---
TEST_DIR = "/kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images"
SAMPLE_SUB = "/kaggle/input/recodai-luc-scientific-image-forgery-detection/sample_submission.csv"
OUT_PATH = "submission.csv"

rows = []
for f in tqdm(sorted(os.listdir(TEST_DIR)), desc="Inference on Test Set"):
    pil = Image.open(Path(TEST_DIR)/f).convert("RGB")
    label, mask, dbg = pipeline_final(pil)  # utilise la version améliorée

    # Sécurisation masque
    if mask is None:
        mask = np.zeros(pil.size[::-1], np.uint8)
    else:
        mask = np.array(mask, dtype=np.uint8)

    # Annotation finale
    if label == "authentic":
        annot = "authentic"
    else:
        annot = rle_encode((mask > 0).astype(np.uint8))

    rows.append({
        "case_id": Path(f).stem,
        "annotation": annot,
        "area": int(dbg.get("area", mask.sum())),
        "mean": float(dbg.get("mean_inside", 0.0)),
        "thr": float(dbg.get("thr", 0.0))
    })


sub = pd.DataFrame(rows)
ss = pd.read_csv(SAMPLE_SUB)
ss["case_id"] = ss["case_id"].astype(str)
sub["case_id"] = sub["case_id"].astype(str)
final = ss[["case_id"]].merge(sub, on="case_id", how="left")
final["annotation"] = final["annotation"].fillna("authentic")
final[["case_id", "annotation"]].to_csv(OUT_PATH, index=False)

print(f"\n✅ Saved submission file: {OUT_PATH}")
print(final.head(10))


sample_files = sorted(os.listdir(TEST_DIR))[:5]
for f in sample_files:
    pil = Image.open(Path(TEST_DIR)/f).convert("RGB")
    label, mask, dbg = pipeline_final(pil)
    mask = np.array(mask, dtype=np.uint8) if mask is not None else np.zeros(pil.size[::-1], np.uint8)

    print(f"{'🔴' if label=='forged' else '🟢'} {f}: {label} | area={mask.sum()} mean={dbg.get('mean_inside', 0):.3f}")

    if label == "authentic":
        plt.figure(figsize=(5,5))
        plt.imshow(pil)
        plt.title(f"{f} — Authentic")
        plt.axis("off")
        plt.show()
    else:
        plt.figure(figsize=(10,5))
        plt.subplot(1,2,1)
        plt.imshow(pil)
        plt.title("Original Image")
        plt.axis("off")
        plt.subplot(1,2,2)
        plt.imshow(pil)
        plt.imshow(mask, alpha=0.45, cmap="Reds")
        plt.title(f"Predicted Forged Mask\nArea={mask.sum()} | Mean={dbg.get('mean_inside', 0):.3f}")
        plt.axis("off")
        plt.show()

## 🔴 Visualizing Predicted Masks with the CNN–DINOv2 Hybrid Model


In [ ]:
import torch, cv2, math, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1️ Use YOUR existing function (already defined in main cell)
@torch.no_grad()
def predict_prob_map(pil):
    """Return DINOv2 segmentation probability map [0,1]."""
    img = pil.resize((IMG_SIZE, IMG_SIZE))
    x = torch.from_numpy(np.array(img, np.float32) / 255.).permute(2, 0, 1)[None].to(device)
    logits = model_seg.forward_seg(x)
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    return prob

# 2️ Use YOUR exact post-processing from main cell
def adaptive_mask(prob, alpha_grad=0.35):
    """Use EXACT same function as in main cell"""
    gx = cv2.Sobel(prob, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(prob, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)
    grad_norm = grad_mag / (grad_mag.max() + 1e-6)
    enhanced = (1 - alpha_grad) * prob + alpha_grad * grad_norm
    enhanced = cv2.GaussianBlur(enhanced, (3,3), 0)
    
    # MUST MATCH: 0.25 * std (from your updated main cell)
    thr = np.mean(enhanced) + 0.25 * np.std(enhanced)
    mask = (enhanced > thr).astype(np.uint8)
    
    # MUST MATCH: 3x3 and 2x2 kernels (from your main cell)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))
    
    return mask, float(thr)

# 3️ Unified visualization pipeline
def pipeline_visual(pil):
    if USE_TTA:
        prob = segment_prob_map_with_tta(pil)  # Your existing function
    else:
        prob = predict_prob_map(pil)
    
    mask, thr = adaptive_mask(prob)
    area = int(mask.sum())
    
    # Fix: Resize mask to match prob dimensions
    mask_small = cv2.resize(mask, (prob.shape[1], prob.shape[0]), interpolation=cv2.INTER_NEAREST)
    mean_inside = float(prob[mask_small == 1].mean()) if area > 0 else 0.0

    # Use SAME thresholds as main cell
    if area < AREA_THR or mean_inside < MEAN_THR:
        label = "authentic"
    else:
        label = "forged"
    
    return label, mask, thr, area, mean_inside

# 4️ Visualization (for validation forged samples)
sample_forged = val_forg[:5]
n = len(sample_forged)
fig, axes = plt.subplots(n, 3, figsize=(12, n * 3))
if n == 1:
    axes = np.expand_dims(axes, axis=0)

for i, p in enumerate(sample_forged):
    pil = Image.open(p).convert("RGB")
    label, m_pred, thr, area, mean = pipeline_visual(pil)

    # Ground Truth mask
    m_gt = np.load(Path(MASK_DIR)/f"{Path(p).stem}.npy")
    if m_gt.ndim == 3: 
        m_gt = np.max(m_gt, axis=0)
    m_gt = (m_gt > 0).astype(np.uint8)

    # Resize all for consistency
    img_disp = cv2.resize(np.array(pil), (IMG_SIZE, IMG_SIZE))
    gt_disp  = cv2.resize(m_gt, (IMG_SIZE, IMG_SIZE))
    pr_disp  = cv2.resize(m_pred, (IMG_SIZE, IMG_SIZE))

    # === Column 1: Original ===
    axes[i, 0].imshow(img_disp)
    axes[i, 0].set_title("🖼️ Original Image", fontsize=11, weight="bold")
    axes[i, 0].axis("off")

    # === Column 2: Ground Truth ===
    axes[i, 1].imshow(gt_disp, cmap="gray")
    axes[i, 1].set_title("✅ Ground Truth", fontsize=11, weight="bold")
    axes[i, 1].axis("off")

    # === Column 3: Predicted Mask ===
    axes[i, 2].imshow(img_disp)
    axes[i, 2].imshow(pr_disp, cmap="coolwarm", alpha=0.45)
    axes[i, 2].set_title(f"🔮 Predicted ({label})\nThr={thr:.3f} | Area={area} | Mean={mean:.3f}",
                         fontsize=10)
    axes[i, 2].axis("off")

plt.subplots_adjust(top=0.92, hspace=0.35)
fig.suptitle("🔍 Segmentation of Forged Samples — CNN–DINOv2 Hybrid", 
             fontsize=16, fontweight="bold", color="#b30000")

plt.show()

# 🟢 Visualization of Authentic Images (Hybrid DINOv2-based Detector)

In [ ]:
import matplotlib.pyplot as plt
import cv2, numpy as np
from pathlib import Path
from PIL import Image

# Select a few authentic examples
sample_auth = val_auth[:5]
n = len(sample_auth)

fig, axes = plt.subplots(n, 2, figsize=(9, n * 3))
if n == 1:
    axes = np.expand_dims(axes, axis=0)

for i, p in enumerate(sample_auth):
    pil = Image.open(p).convert("RGB")
    label, m_pred, thr, area, mean = pipeline_visual(pil)  # <-- version alignée avec ta nouvelle pipeline

    # Predicted mask (should be empty for authentic images)
    m_pred = (m_pred > 0).astype(np.uint8) if m_pred is not None else np.zeros((IMG_SIZE, IMG_SIZE))

    # Resize for consistent display
    img_disp = cv2.resize(np.array(pil), (IMG_SIZE, IMG_SIZE))
    pr_disp  = cv2.resize(m_pred, (IMG_SIZE, IMG_SIZE))

    # === Column 1: Original Image ===
    axes[i, 0].imshow(img_disp)
    axes[i, 0].set_title("🖼️ Original Image", fontsize=11, weight="bold")
    axes[i, 0].axis("off")

    # === Column 2: Predicted Mask ===
    axes[i, 1].imshow(img_disp)
    axes[i, 1].imshow(pr_disp, cmap="coolwarm", alpha=0.45)
    axes[i, 1].set_title(
        f"🟢 Predicted: {label.upper()}\nArea={area} | Mean={mean:.3f} | Thr={thr:.3f}",
        fontsize=10
    )
    axes[i, 1].axis("off")

    for j in range(2):
        axes[i, j].set_aspect("equal")

plt.subplots_adjust(top=0.90, hspace=0.35)
fig.suptitle("🟢 Segmentation of Authentic Images — CNN–DINOv2 Hybrid",
             fontsize=16, fontweight="bold", color="#009933")
plt.show()

In [ ]:
# Add this cell to test on forged images
print("\n" + "="*50)
print("TESTING ON FORGED VALIDATION IMAGES")
print("="*50)

val_forged_items = [(p, 1) for p in val_forg[:10]]
forged_results = []
for p,_ in tqdm(val_forged_items, desc="Forged Validation"):
    pil = Image.open(p).convert("RGB")
    label, m_pred, dbg = pipeline_final(pil)
    m_gt = np.load(Path(MASK_DIR)/f"{Path(p).stem}.npy")
    if m_gt.ndim==3: m_gt=np.max(m_gt,axis=0)
    m_gt=(m_gt>0).astype(np.uint8)
    m_pred=(m_pred>0).astype(np.uint8) if m_pred is not None else np.zeros_like(m_gt)
    f1 = f1_score(m_gt.flatten(), m_pred.flatten(), zero_division=0)
    forged_results.append((Path(p).stem, f1, dbg, label))

print("\nForged Image Results:")
for cid,f1,dbg,label in forged_results:
    print(f"{cid} — {label} | F1={f1:.4f} | area={dbg['area']} mean={dbg['mean_inside']:.3f}")

avg_f1 = np.mean([r[1] for r in forged_results])
print(f"\nAverage F1 on forged images: {avg_f1:.4f}")